# S11 — physics-based dose augmentation for the benign/malignant classifier (no generative model)
Rules: `S11_preregistration.txt` (fixed before any S11 code ran). ConvNeXt-Tiny, fixed S2 recipe, 5 seeds × 3 arms ×
2 training datasets (CMMD, CBIS-DDSM):
**R0** standard augmentation · **R1** + generic white noise (matched power) · **R2** + physics dose-reduction noise.
Primary endpoints P1–P3 on CMMD (clean and simulated 25 %-dose test images), secondary cross-dataset tests.

**Input:** Add Input → the **S8 replication** notebook output (latest version; it contains `cmmd_data/` and `clean_data/`).
GPU T4 ×2 · Internet ON (ImageNet weights) · Save & Run All (Commit). **≈ 1.5 h.** Send back `S11_results.zip`.

In [ ]:
# ===================== CELL 1 — config + code + inputs =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
OUT = "/kaggle/working/S11"; WD = "/kaggle/working/s11_code"
os.makedirs(OUT, exist_ok=True); os.makedirs(WD, exist_ok=True)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA", "CMMD")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(os.path.join(os.environ.get("PA_TMP", "/tmp"), f"{split}_cache.npz"))      # S8: dataset switch
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(os.path.join(os.environ.get("PA_DATA", "/kaggle/working/clean_data"), f"{split}_cache.npz"))   # S8: dataset switch
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "pinn_physics.py": r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "s11_clf.py": r'''# s11_clf.py - S11: physics-based dose augmentation for a benign/malignant classifier (rules: S11_preregistration.txt).
# One call = one training dataset, a list of arms and seeds. Trains the fixed S2 classifier recipe with
#   R0 standard augmentation | R1 + white noise of matched power | R2 + physics dose-reduction noise,
# then saves predicted probabilities on every test condition (clean, dose50, dose25 on CMMD; clean on CBIS).
import os, sys, json, math, time, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from a6_classifier import build, prep, augment, predict, auc, dev
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--train_ds", choices=["cmmd", "cbis"], required=True)
ap.add_argument("--arms", default="R0,R1,R2")
ap.add_argument("--seeds", default="0,1,2,3,4")
ap.add_argument("--epochs", type=int, default=15)
ap.add_argument("--data_cmmd", required=True)      # folder with train_cache.npz / test_cache.npz (with patient ids)
ap.add_argument("--data_cbis", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args()
os.makedirs(a.out, exist_ok=True)
TAG = f"S11-{a.train_ds}"
def log(*m): print(f"[{TAG}]", *m, flush=True)

def val_mask(patients, frac=0.15, seed=0):            # identical to phaseA_common.val_mask
    ups = sorted(set(patients)); rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])
def load(folder, split):
    d = np.load(os.path.join(folder, f"{split}_cache.npz")); return d["x"], d["y"].astype(np.int64), d["patient"].astype(str)

xtr, ytr, ptr = load(a.data_cmmd if a.train_ds == "cmmd" else a.data_cbis, "train")
vm = val_mask(ptr); XF, YF, XV, YV = xtr[~vm], ytr[~vm], xtr[vm], ytr[vm]
xc, yc, _ = load(a.data_cmmd, "test"); xb_, yb_, _ = load(a.data_cbis, "test")
if a.smoke:
    XF, YF, a.epochs = XF[:64], YF[:64], 1
    xc, yc, xb_, yb_ = xc[:40], yc[:40], xb_[:40], yb_[:40]

def mm01(x):                                          # (B,1,H,W) -> per-image min-max [0,1]
    mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return (x - mn) / (mx - mn + 1e-8)

# ---------------- noise model: linear Poisson-Gaussian NLF of THIS training set (PhysicsNLF, as in S7/S8) ----------
def fit_nlf(X_u8):
    X = torch.from_numpy(X_u8).float(); mn = X.amin(dim=(1, 2), keepdim=True); mx = X.amax(dim=(1, 2), keepdim=True)
    U = ((X - mn) / (mx - mn + 1e-8) * 255).round().clamp(0, 255).to(torch.uint8)
    rep = PhysicsNLF(dev).fit(U); lin = rep["linear_poisson_gaussian_only"]
    return float(lin["a"]), float(lin["b"]), float(lin["R2_var"])
A_, B_, R2_ = fit_nlf(XF)
g1 = torch.exp(-(torch.arange(7.0) - 3) ** 2 / 2); g1 = g1 / g1.sum(); G = torch.outer(g1, g1)
delta = torch.zeros(7, 7); delta[3, 3] = 1
K_HP = float(((delta - G) ** 2).sum())               # E[highpass^2] = K_HP * white-noise variance
log(f"train {len(YF)} (benign {int((YF == 0).sum())}) | val {len(YV)} | NLF sigma^2 = {A_:.3e} I + {B_:.3e} (R2 {R2_:.2f}) | k_hp {K_HP:.4f}")

def phys_var(I, alpha):
    al = alpha.view(-1, 1, 1, 1)
    return ((A_ * I * (1 / al - 1) + B_ * (1 / al ** 2 - 1)) / K_HP).clamp_min(0)
def degrade(I, alpha, g):
    return (I + phys_var(I, alpha).sqrt() * torch.randn(I.shape, generator=g, device=I.device)).clamp(0, 1)

# matched power for R1: mean added R2 variance over 200 training images (Monte Carlo, fixed seed)
with torch.no_grad():
    gm = torch.Generator(device=dev).manual_seed(99)
    I = mm01(torch.from_numpy(XF[:200]).to(dev).float().unsqueeze(1))
    al = 0.25 + 0.75 * torch.rand(I.shape[0], generator=gm, device=dev)
    V_GEN = float(phys_var(I, al).mean())
log(f"R1 generic white-noise variance (matched to R2 mean) = {V_GEN:.3e} (std {math.sqrt(V_GEN) * 255:.1f} grey levels)")

def noise_aug(x01, arm, g):
    if arm == "R0":
        return x01
    B = x01.shape[0]; I = mm01(x01)
    use = (torch.rand(B, generator=g, device=dev) < 0.8).view(-1, 1, 1, 1)
    if arm == "R1":
        out = (I + math.sqrt(V_GEN) * torch.randn(I.shape, generator=g, device=dev)).clamp(0, 1)
    else:
        out = degrade(I, 0.25 + 0.75 * torch.rand(B, generator=g, device=dev), g)
    return torch.where(use, out, x01)

def train(arm, seed, epochs, bs=32):                  # = a6_classifier.train_clf + the noise hook
    Xg, Yg = torch.from_numpy(XF).to(dev), torch.from_numpy(YF).long().to(dev)
    w = torch.tensor([len(YF) / (2 * max(1, (YF == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed); m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(YF) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda"); g = torch.Generator(device=dev).manual_seed(seed)
    gn = torch.Generator(device=dev).manual_seed(seed * 7 + 1)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(YF), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev); tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = noise_aug(augment(Xg[idx].float().div(255).unsqueeze(1), g), arm, gn)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss.detach())
        if (ep + 1) % 5 == 0 or ep == 0:
            log(f"{arm} seed {seed} epoch {ep + 1}/{epochs} loss {tl / spe:.4f}")
    return m

# fixed degraded CMMD test sets (alpha 0.5 and 0.25, seed 4242) - built once, identical for every arm and seed
with torch.no_grad():
    gt = torch.Generator(device=dev).manual_seed(4242); TESTS = {"cmmd_clean": xc, "cbis_clean": xb_}
    for al in (0.5, 0.25):
        D = []
        for i in range(0, len(xc), 64):
            I = mm01(torch.from_numpy(xc[i:i + 64]).to(dev).float().unsqueeze(1))
            D.append((degrade(I, torch.full((I.shape[0],), al, device=dev), gt) * 255).round().byte().squeeze(1).cpu().numpy())
        TESTS[f"cmmd_dose{int(al * 100)}"] = np.concatenate(D)
LAB = {"cmmd_clean": yc, "cbis_clean": yb_, "cmmd_dose50": yc, "cmmd_dose25": yc}
json.dump({"train_ds": a.train_ds, "nlf_a": A_, "nlf_b": B_, "nlf_R2": R2_, "k_hp": K_HP, "v_generic": V_GEN,
           "n_train": int(len(YF)), "n_val": int(len(YV))}, open(os.path.join(a.out, f"s11_{a.train_ds}_meta.json"), "w"), indent=1)
for arm in a.arms.split(","):
    for seed in [int(s) for s in a.seeds.split(",")]:
        f = os.path.join(a.out, f"pred_{a.train_ds}_{arm}_s{seed}.npz")
        if os.path.exists(f):
            log(f"{f} exists -> skipped"); continue
        t0 = time.time(); m = train(arm, seed, a.epochs).eval()
        P = {k: predict(m, v) for k, v in TESTS.items()}; P["val"] = predict(m, XV)
        np.savez(f, **P, **{f"y_{k}": v for k, v in LAB.items()}, y_val=YV)
        log(f"{arm} seed {seed}: val AUC {auc(P['val'], YV):.3f} | " + " | ".join(f"{k} {auc(P[k], LAB[k]):.3f}" for k in TESTS)
            + f" | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
''',
         "s11_stats.py": r'''# s11_stats.py - S11 endpoints (pure numpy/scipy; rules in S11_preregistration.txt).
import os, sys, glob, json
import numpy as np
from scipy.stats import rankdata

D = sys.argv[1]
def auc(p, y):
    r = rankdata(p); n1 = (y == 1).sum(); n0 = (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def load(ds, arm):
    fs = sorted(glob.glob(f"{D}/pred_{ds}_{arm}_s*.npz"))
    return [dict(np.load(f)) for f in fs]
def ens(runs, cond):
    return np.mean([r[cond] for r in runs], 0), runs[0][f"y_{cond}"]
rng = np.random.default_rng(0)
def boot_diff(pa, pb, y, alpha, n=2000):
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]; d = []
    for _ in range(n):
        b = np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]); d.append(auc(pa[b], y[b]) - auc(pb[b], y[b]))
    return float(np.quantile(d, alpha / 2)), float(np.quantile(d, 1 - alpha / 2))
L = ["S11 REPORT - physics dose augmentation for the benign/malignant classifier (rules: S11_preregistration.txt)", "=" * 100]
R = {}
for ds in ("cmmd", "cbis"):
    runs = {arm: load(ds, arm) for arm in ("R0", "R1", "R2")}
    if not all(runs.values()):
        L.append(f"{ds}-trained: incomplete ({ {k: len(v) for k, v in runs.items()} })"); continue
    L.append(f"\n{ds.upper()}-TRAINED models ({len(runs['R0'])} seeds per arm) - single-seed AUC mean +- SD | 5-seed ensemble AUC")
    for cond in ("val", "cmmd_clean", "cmmd_dose50", "cmmd_dose25", "cbis_clean"):
        row = []
        for arm in ("R0", "R1", "R2"):
            s = [auc(r[cond], r[f"y_{cond}"] if cond != "val" else r["y_val"]) for r in runs[arm]]
            pe = np.mean([r[cond] for r in runs[arm]], 0); ye = runs[arm][0][f"y_{cond}"] if cond != "val" else runs[arm][0]["y_val"]
            row.append(f"{arm} {np.mean(s):.3f}+-{np.std(s, ddof=1):.3f} | {auc(pe, ye):.3f}")
            R.setdefault(ds, {}).setdefault(cond, {})[arm] = {"seed_aucs": s, "ensemble_auc": auc(pe, ye)}
        L.append(f"  {cond:12s}: " + "   ".join(row))
def endpoint(name, ds, cond, A, B, alpha, kind="sup"):
    pa, y = ens(load(ds, A), cond); pb, _ = ens(load(ds, B), cond)
    d = auc(pa, y) - auc(pb, y); lo, hi = boot_diff(pa, pb, y, alpha)
    sa = [auc(r[cond], r[f"y_{cond}"]) for r in load(ds, A)]; sb = [auc(r[cond], r[f"y_{cond}"]) for r in load(ds, B)]
    same = int(sum(np.sign(x - z) == np.sign(d) for x, z in zip(sa, sb)))
    ok = (lo > 0) if kind == "sup" else (lo > -0.02)
    excl = (lo > 0) or (hi < 0)                       # descriptive: CI excludes 0 in EITHER direction (fix 2026-10-23)
    L.append(f"  {name:4s} {ds}-trained, {cond:12s}: AUC {A} - {B} = {d:+.4f}  CI{100 * (1 - alpha):.2f}% {lo:+.4f}..{hi:+.4f} "
             f"| same sign in {same}/{len(sa)} seeds -> {('PASS' if ok else 'fail') if alpha < 0.05 else ('CI excludes 0' if excl else 'CI includes 0')}" + (" (non-inferiority, margin -0.02)" if kind == "ni" else ""))
    return {"diff": d, "ci": [lo, hi], "same_sign_seeds": same, "passed": bool(ok)}
if "cmmd" in R:
    L.append("\nPRIMARY ENDPOINTS (98.33 % CI, Bonferroni over 3)")
    E = {"P1": endpoint("P1", "cmmd", "cmmd_dose25", "R2", "R0", 0.05 / 3),
         "P2": endpoint("P2", "cmmd", "cmmd_dose25", "R2", "R1", 0.05 / 3),
         "P3": endpoint("P3", "cmmd", "cmmd_clean", "R2", "R0", 0.05 / 3, "ni")}
    L.append("\nSECONDARY (95 % CI, descriptive)")
    for nm, ds, cond, A, B in (("S1a", "cmmd", "cmmd_dose50", "R2", "R0"), ("S1b", "cmmd", "cmmd_dose50", "R2", "R1"),
                               ("S2a", "cmmd", "cbis_clean", "R2", "R0"), ("S2b", "cmmd", "cbis_clean", "R1", "R0"),
                               ("S3a", "cbis", "cmmd_clean", "R2", "R0"), ("S3b", "cbis", "cmmd_clean", "R1", "R0"),
                               ("S4", "cbis", "cbis_clean", "R2", "R0"), ("S5", "cmmd", "cmmd_dose25", "R1", "R0")):
        if ds in R:
            E[nm] = endpoint(nm, ds, cond, A, B, 0.05)
    R["endpoints"] = E
txt = "\n".join(L); print(txt)
open(f"{D}/S11_report.txt", "w").write(txt + "\n"); json.dump(R, open(f"{D}/S11_results.json", "w"), indent=1, default=float)
'''}
for n, s in FILES.items():
    Path(f"{WD}/{n}").write_text(s)
def walk_find(name, root="/kaggle/input"):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git", "ckpt")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)
def need_dir(summary, parent):
    h = [os.path.dirname(p) for p in walk_find(summary) if os.path.basename(os.path.dirname(p)) == parent]
    if not h:
        raise SystemExit(f"!! {parent}/{summary} not found -> Add Input -> the S8 replication notebook output")
    print(parent, "->", h[0]); return h[0]
DATA_CMMD = need_dir("cmmd_summary.json", "cmmd_data"); DATA_CBIS = need_dir("clean_summary.json", "clean_data")
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU)
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True); RC = {}
def run_one(name, gpu, argv):
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
    env["PA_OUT"] = f"{OUT}/tmp"
    print(f">>> {name} started on GPU {env['CUDA_VISIBLE_DEVICES']}", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush(); print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]}" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)
COMMON = ["--data_cmmd", DATA_CMMD, "--data_cbis", DATA_CBIS]
print("Cell 1 done.")

In [ ]:
# ===================== CELL 2 — smoke test (~5 min) =====================
SM = "/kaggle/working/s11_smoke"
run_one("smoke", 0, ["s11_clf.py", "--train_ds", "cmmd", "--arms", "R0,R1,R2", "--seeds", "0", "--smoke", "--out", SM] + COMMON)
if RC.get("smoke") != 0 or len(glob.glob(f"{SM}/pred_*.npz")) != 3:
    raise SystemExit("SMOKE TEST FAILED -> send Claude this block + S11/logs/smoke.log")
shutil.rmtree(SM, ignore_errors=True); print("SMOKE TEST PASSED")

In [ ]:
# ===================== CELL 3 — training: CMMD-trained on GPU 0, CBIS-trained on GPU 1 (~1-1.5 h) =====================
ths = [threading.Thread(target=run_one, args=(f"train_{ds}", g, ["s11_clf.py", "--train_ds", ds, "--out", OUT] + COMMON))
       for g, ds in enumerate(("cmmd", "cbis"))]
for t in ths: t.start()
for t in ths: t.join()

In [ ]:
# ===================== CELL 4 — endpoints + zip =====================
r = subprocess.run([sys.executable, "s11_stats.py", OUT], cwd=WD, capture_output=True, text=True)
print(r.stdout[-20000:]); print(r.stderr[-3000:] if r.returncode else "")
json.dump(RC, open(f"{OUT}/return_codes.json", "w"), indent=1)
shutil.rmtree(f"{OUT}/tmp", ignore_errors=True)
z = shutil.make_archive("/kaggle/working/S11_results", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | return codes {RC}")